# 02 - Landing Zone CSV para Bronze Delta Lake

Este notebook le os arquivos CSV do bucket **landing-zone** no MinIO e grava cada tabela em formato **Delta Lake** no bucket **bronze**.

Fluxo executado:

`MinIO / landing-zone/*.csv -> Spark -> MinIO / bronze/<tabela> em Delta Lake`

## 1. Imports e variaveis

In [1]:
import os

import boto3
from botocore.client import Config
from delta.tables import DeltaTable
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv(override=True)

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
BRONZE_BUCKET = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

tables = ['clientes', 'produtos', 'pedidos', 'itens_pedido']

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')
print(f'Tabelas: {tables}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze
Tabelas: ['clientes', 'produtos', 'pedidos', 'itens_pedido']


## 2. Criar SparkSession com Delta Lake e MinIO

In [2]:
spark = (
    SparkSession.builder
    .appName('Landing Zone CSV to Bronze Delta Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession criada com suporte a Delta Lake e MinIO.')

26/05/05 20:09:14 WARN Utils: Your hostname, And resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/05 20:09:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/anderson/git-clone/spark-delta-minio/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/anderson/.ivy2/cache
The jars for the packages stored in: /home/anderson/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fa26c884-7704-4267-9166-e4fc273d1534;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 413ms :: artifacts dl 13ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.

SparkSession criada com suporte a Delta Lake e MinIO.


## 3. Criar e limpar bucket bronze

In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe.')
except Exception:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado.')

response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET)
objetos = [{'Key': obj['Key']} for obj in response.get('Contents', [])]

if objetos:
    s3_client.delete_objects(Bucket=BRONZE_BUCKET, Delete={'Objects': objetos})
    print(f'{len(objetos)} objeto(s) removido(s) do bucket [{BRONZE_BUCKET}].')
else:
    print(f'Bucket [{BRONZE_BUCKET}] ja estava vazio.')

Bucket [bronze] criado.
Bucket [bronze] ja estava vazio.


## 4. Validar arquivos CSV no landing-zone

In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
landing_keys = sorted(obj['Key'] for obj in response.get('Contents', []))
expected_keys = [f'{table}.csv' for table in tables]

print(f'Arquivos encontrados no bucket [{LANDING_BUCKET}]:')
for key in landing_keys:
    print(f'  - {key}')

missing_keys = [key for key in expected_keys if key not in landing_keys]
if missing_keys:
    raise FileNotFoundError(f'Arquivos ausentes no landing-zone: {missing_keys}')

print('Todos os CSVs esperados foram encontrados no landing-zone.')

Arquivos encontrados no bucket [landing-zone]:
  - clientes.csv
  - itens_pedido.csv
  - pedidos.csv
  - produtos.csv
Todos os CSVs esperados foram encontrados no landing-zone.


## 5. Ler CSVs e gravar tabelas Delta no bronze

In [5]:
for table in tables:
    print(f'Convertendo tabela {table} para Delta Lake...')

    input_path = f's3a://{LANDING_BUCKET}/{table}.csv'
    output_path = f's3a://{BRONZE_BUCKET}/{table}'

    df = (
        spark.read
        .option('header', 'true')
        .option('inferSchema', 'true')
        .csv(input_path)
    )

    total = df.count()
    print(f'Registros lidos de {table}: {total}')
    df.printSchema()

    (
        df.write
        .format('delta')
        .mode('overwrite')
        .save(output_path)
    )

    print(f'Tabela {table} salva em Delta Lake em: {output_path}\n')

print('Conversao para Delta Lake concluida.')

Convertendo tabela clientes para Delta Lake...


26/05/05 20:09:22 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Registros lidos de clientes: 100
root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- telefone: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_cadastro: date (nullable = true)



Tabela clientes salva em Delta Lake em: s3a://bronze/clientes

Convertendo tabela produtos para Delta Lake...
Registros lidos de produtos: 50
root
 |-- id: integer (nullable = true)
 |-- nome_produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- preco: double (nullable = true)
 |-- estoque: integer (nullable = true)
 |-- ativo: boolean (nullable = true)

Tabela produtos salva em Delta Lake em: s3a://bronze/produtos

Convertendo tabela pedidos para Delta Lake...
Registros lidos de pedidos: 200
root
 |-- id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- data_pedido: date (nullable = true)
 |-- status: string (nullable = true)
 |-- valor_total: double (nullable = true)

Tabela pedidos salva em Delta Lake em: s3a://bronze/pedidos

Convertendo tabela itens_pedido para Delta Lake...
Registros lidos de itens_pedido: 400
root
 |-- id: integer (nullable = true)
 |-- pedido_id: integer (nullable = true)
 |-- produto_id: integer (nullable = tr

## 6. Validar tabelas Delta no bronze

In [6]:
for table in tables:
    path = f's3a://{BRONZE_BUCKET}/{table}'

    print(f'Validando tabela Delta: {table}')
    is_delta = DeltaTable.isDeltaTable(spark, path)
    print(f'Eh Delta Lake: {is_delta}')

    df_delta = (
        spark.read
        .format('delta')
        .load(path)
    )

    print(f'Total de registros em {table}: {df_delta.count()}')
    df_delta.show(5, truncate=False)

    if not is_delta:
        raise RuntimeError(f'A tabela {table} nao foi gravada como Delta Lake.')

Validando tabela Delta: clientes
Eh Delta Lake: True


26/05/05 20:09:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Total de registros em clientes: 100


+---+-----------------+-------------------------+------------+----------+--------------+-------------+
|id |nome             |email                    |telefone    |cidade    |estado        |data_cadastro|
+---+-----------------+-------------------------+------------+----------+--------------+-------------+
|1  |Janeva California|jcalifornia0@github.io   |919-558-3498|Kamimaruko|NULL          |2025-09-11   |
|2  |Margaretha Eplate|meplate1@businessweek.com|725-750-3698|Monsanto  |Castelo Branco|2025-06-03   |
|3  |Niall Ellerton   |nellerton2@sakura.ne.jp  |101-148-8590|Daohe     |NULL          |2026-03-26   |
|4  |Lana Scorrer     |lscorrer3@soundcloud.com |696-778-3566|Kauit     |NULL          |2026-04-29   |
|5  |Enos Aston       |easton4@intel.com        |492-788-0686|Medellin  |NULL          |2025-10-04   |
+---+-----------------+-------------------------+------------+----------+--------------+-------------+
only showing top 5 rows

Validando tabela Delta: produtos
Eh Delta Lake: 

Total de registros em produtos: 50
+---+----------------------+----------------------+-----+-------+-----+
|id |nome_produto          |categoria             |preco|estoque|ativo|
+---+----------------------+----------------------+-----+-------+-----+
|1  |Insulated Cooler      |Outdoor               |39.99|1      |false|
|2  |Organic Baby Carrots  |Food - Fresh Produce  |2.99 |2      |true |
|3  |Hibiscus Tea Bags     |Food - Beverages      |3.79 |3      |false|
|4  |Spinach Artichoke Dip |Food - Snacks         |4.99 |4      |true |
|5  |Knitted Infinity Scarf|Clothing - Accessories|29.99|5      |true |
+---+----------------------+----------------------+-----+-------+-----+
only showing top 5 rows

Validando tabela Delta: pedidos
Eh Delta Lake: True


Total de registros em pedidos: 200
+---+----------+-----------+---------+-----------+
|id |cliente_id|data_pedido|status   |valor_total|
+---+----------+-----------+---------+-----------+
|1  |63        |2026-03-08 |PAGO     |3.49       |
|2  |17        |2025-12-21 |CANCELADO|9.99       |
|3  |87        |2025-06-07 |ENTREGUE |2.99       |
|4  |71        |2025-06-07 |ENTREGUE |2.39       |
|5  |26        |2025-06-03 |PENDENTE |39.99      |
+---+----------+-----------+---------+-----------+
only showing top 5 rows

Validando tabela Delta: itens_pedido
Eh Delta Lake: True


Total de registros em itens_pedido: 400


[Stage 83:=====================================================>  (48 + 2) / 50]

+---+---------+----------+----------+--------------+
|id |pedido_id|produto_id|quantidade|preco_unitario|
+---+---------+----------+----------+--------------+
|1  |184      |30        |7         |39.99         |
|2  |185      |15        |7         |2.49          |
|3  |101      |50        |7         |2.99          |
|4  |169      |6         |10        |12.99         |
|5  |118      |35        |5         |2.99          |
+---+---------+----------+----------+--------------+
only showing top 5 rows



## 7. Validar _delta_log no bucket bronze

In [7]:
for table in tables:
    prefix = f'{table}/_delta_log/'
    response = s3_client.list_objects_v2(Bucket=BRONZE_BUCKET, Prefix=prefix)
    delta_log_files = response.get('Contents', [])

    print(f'{table}: {len(delta_log_files)} arquivo(s) em {BRONZE_BUCKET}/{prefix}')

    if not delta_log_files:
        raise RuntimeError(f'_delta_log nao encontrado para a tabela {table}.')

clientes: 2 arquivo(s) em bronze/clientes/_delta_log/
produtos: 2 arquivo(s) em bronze/produtos/_delta_log/
pedidos: 2 arquivo(s) em bronze/pedidos/_delta_log/
itens_pedido: 2 arquivo(s) em bronze/itens_pedido/_delta_log/


## 8. Historico Delta

In [8]:
for table in tables:
    path = f's3a://{BRONZE_BUCKET}/{table}'

    print(f'Historico Delta da tabela: {table}')
    history_df = spark.sql(f'DESCRIBE HISTORY delta.`{path}`')
    history_df.show(truncate=False)

Historico Delta da tabela: clientes
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                             |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-05-05 20:09:38|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |false 

## 9. Encerrar Spark

In [9]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
